# Process Window — Dose-Focus Matrix

This tutorial computes a **Bossung plot** (CD vs dose × focus) and extracts
**Depth of Focus (DoF)** and **Exposure Latitude (EL)**.

The process window defines the range of dose and focus that keeps CD within
±10 % of the target.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

from euv.pipeline import SimulationConfig, run_simulation

In [ ]:
# Dose-focus grid
doses = np.linspace(10, 40, 7)      # 10 to 40 mJ/cm²
focuses = np.linspace(-50, 50, 7)   # -50 to +50 nm defocus
target_cd = 32.0

cd_matrix = np.zeros((len(doses), len(focuses)))

for i, d in enumerate(doses):
    for j, f in enumerate(focuses):
        cfg = SimulationConfig(
            period_nm=64.0,
            line_width_nm=target_cd,
            dose_mj_cm2=d,
            focus_nm=f,
            na=0.33,
            grid=256,
        )
        result = run_simulation(cfg)
        cd_matrix[i, j] = result.cd_nm
        print(f"d={d:.0f}, f={f:+3.0f} → CD={result.cd_nm:.2f}")

In [ ]:
# Bossung plot
plt.figure(figsize=(10, 6))
for i, d in enumerate(doses):
    plt.plot(focuses, cd_matrix[i], 'o-', label=f'{d:.0f} mJ/cm²')

plt.axhline(target_cd * 0.9, color='gray', ls='--', alpha=0.5, label='±10 % spec')
plt.axhline(target_cd * 1.1, color='gray', ls='--', alpha=0.5)
plt.xlabel('Defocus [nm]')
plt.ylabel('CD [nm]')
plt.title('Bossung Plot — CD vs Focus at Various Doses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Process window metrics
lo = target_cd * 0.9
hi = target_cd * 1.1
in_spec = (cd_matrix >= lo) & (cd_matrix <= hi)

# Depth of focus (max focus range at any dose)
dof = 0.0
for i in range(len(doses)):
    f_ok = focuses[in_spec[i]]
    if len(f_ok) > 1:
        dof = max(dof, f_ok.max() - f_ok.min())

# Exposure latitude at best focus
j_best = int(np.argmax(in_spec.sum(axis=0)))
d_ok = doses[in_spec[:, j_best]]
el = (d_ok.max() - d_ok.min()) / target_cd * 100 if len(d_ok) > 1 else 0.0

print(f"Target CD:     {target_cd:.0f} nm")
print(f"Depth of Focus: {dof:.1f} nm")
print(f"Exposure Latitude: {el:.1f} %")
print(f"Best-focus plane: {focuses[j_best]:+.0f} nm")